In [ ]:
# Library code lives in kandi/; this notebook only runs demos.
from pathlib import Path

from kandi import (
    build_graph_context,
    compute_max_flow,
    draw_road_graph,
    extract_customer_driver_flow_matching,
    load_bipartite_graphs_from_file,
    postprocess_matching_metrics,
    print_flow_matching,
    print_graph_context,
    run_complex_test,
    run_complex_test_sweep,
    timestamped_metrics_csv_path,
    write_pipeline_results_csv,
)

In [ ]:
# Load the default dataset.
GRAPH_PATH = Path("1_heavy.graph")
graphs = load_bipartite_graphs_from_file(GRAPH_PATH)

In [ ]:
# Pick one graph and build its bipartite context.
GRAPH_INDEX = 32
ctx = build_graph_context(graphs[GRAPH_INDEX])
G_road, SOURCE, SINK = ctx["G_road"], ctx["SOURCE"], ctx["SINK"]
part, road_nodes = ctx["part"], ctx["road_nodes"]
print_graph_context(ctx, GRAPH_INDEX)

In [ ]:
draw_road_graph(ctx)

In [ ]:
# Max flow on the selected graph, then read off the matching.
max_flow, G_cap, G_residual, ff_sec = compute_max_flow(G_road, SOURCE, SINK)
print(
    f"Max flow from source (id {SOURCE}) to sink (id {SINK}): {max_flow} "
    f"(Ford-Fulkerson loop {ff_sec * 1000:.4f} ms)"
)

matching = extract_customer_driver_flow_matching(
    G_road, G_cap, G_residual, part, road_nodes
)
metrics = postprocess_matching_metrics(matching, ff_sec)
print_flow_matching(matching, G_road)
print(
    f"Metrics: coverage {metrics['coverage_pct']:.2f}%, "
    f"avg customers/driver {metrics['avg_customers_per_driver']:.4f}, "
    f"avg drivers/customer {metrics['avg_drivers_per_customer']:.4f}, "
    f"Ford-Fulkerson {metrics['ford_fulkerson_seconds'] * 1000:.4f} ms"
)

In [ ]:
# Generate a complex bipartite graph and run the full pipeline on it.
complex_result = run_complex_test(
    graph_id=100001,
    n_customers=7047,
    n_drivers=97,
    edges_per_customer=(3, 6),
    graph_seed=42,
    customer_capacity=2.0,         # raise to let one customer reach multiple drivers
    driver_capacity=None,          # None => unlimited customers per driver
    scheduling_ride_duration=(5.0, 30.0),  # per-customer R_i ~ U[lo, hi]
    scheduling_max_resolve_rounds=5,
    plot=False,
    verbose=True,
)
csv_path = timestamped_metrics_csv_path("complex_test_metrics")
write_pipeline_results_csv([complex_result], csv_path)
print(f"\nWrote per-graph metrics to {csv_path.resolve()}")

In [2]:
# Sweep (n_customers, n_drivers) configs; metrics go to one CSV.
sweep_results = run_complex_test_sweep(
    pairs=[(1000, 200)],
    customer_range=(6000, 8000, 1000),
    driver_range=(100, 150, 10),
    edges_per_customer=(3, 6),
    customer_capacity=5.0,
    driver_capacity=None,
    scheduling_ride_duration=(5.0, 30.0),
    scheduling_max_resolve_rounds=5,
    plot=False,
    verbose=True,
)
print(f"\nSweep finished: {len(sweep_results)} configs.")

[sweep] running 19 (n_customers, n_drivers) configs
[sweep] === config 1/19: customers=1000, drivers=200 ===
Loaded 1 graphs from /Users/aapo.laakkio/koulu/kandi/graphs/complex_09e704960bef44a4aaec5873b35a9a1c.graph
Kept 1 bipartite graphs; discarded 0 non-bipartite (ids: [])
Each kept graph: super-source (customers, partition 0) and super-sink (drivers, partition 1) added; entry['source'] / entry['sink'] are those terminal ids.
[pipeline] >>> processing graph index=0 id=1
Graph id=1 (index 0), |V|=1202, |E|=5738, super-source=1200, super-sink=1201
Mapping: color 0 = customers (1000 road nodes), color 1 = drivers (200 road nodes); super-source connects to customers, super-sink to drivers.
[pipeline] running Ford-Fulkerson max flow...
[pipeline]   max flow = 4280 (FF 1808.24 ms)
[pipeline] extracting customer<->driver matching...
[pipeline]   matching: coverage=100.0% (1000/1000 served), avg drivers/customer=4.28, avg customers/driver=21.40
[scheduling] generating instance: N=1000, K=20